In [0]:
%pip install alpaca-py

Looking in indexes: [REDACTED]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
import json
import os
import shutil
from datetime import datetime, timedelta, timezone

import pandas as pd
from alpaca.data.historical import CryptoHistoricalDataClient
from alpaca.data.requests import CryptoBarsRequest
from alpaca.data.timeframe import TimeFrame, TimeFrameUnit

SYMBOL = "BTC/USD"
VOLUME_DIR = "/Volumes/workspace/bitcoin/raw/alpaca/BTC_USD/4h"

now = datetime.now(timezone.utc)
this_month = datetime(now.year, now.month, 1, tzinfo=timezone.utc)
# Start on the first day of the previous month, so both monthly files are always complete
start = (this_month - timedelta(days=1)).replace(day=1)

client = CryptoHistoricalDataClient()
request = CryptoBarsRequest(
    symbol_or_symbols=[SYMBOL],
    timeframe=TimeFrame(4, TimeFrameUnit.Hour),
    start=start,
    end=now,
)
df = client.get_crypto_bars(request).df.reset_index()
df["month"] = df["timestamp"].dt.strftime("%Y-%m")
print("rows pulled:", len(df), "| first:", df["timestamp"].min(), "| last:", df["timestamp"].max())

os.makedirs(VOLUME_DIR, exist_ok=True)

# One file per month, nothing cleaned. Each file is built locally, then copied across
for month, part in df.groupby("month"):
    local = f"/tmp/{month}.parquet"
    part.drop(columns="month").to_parquet(local, index=False, coerce_timestamps="us", allow_truncated_timestamps=True)
    shutil.copy(local, f"{VOLUME_DIR}/{month}.parquet")
    print("landed", month, len(part), "rows")

manifest = {
    "source": "alpaca",
    "symbol": SYMBOL,
    "timeframe": "4Hour",
    "pulled_at_utc": now.isoformat(),
    "rows_pulled": len(df),
    "first_bar": str(df["timestamp"].min()),
    "last_bar": str(df["timestamp"].max()),
}
with open("/tmp/_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)
shutil.copy("/tmp/_manifest.json", f"{VOLUME_DIR}/_manifest.json")
print(manifest)

rows pulled: 245 | first: 2026-10-01 00:00:00+00:00 | last: 2026-11-10 16:00:00+00:00
landed 2026-10 186 rows
landed 2026-11 59 rows
{'source': 'alpaca', 'symbol': 'BTC/USD', 'timeframe': '4Hour', 'pulled_at_utc': '2026-11-10T18:19:49.319909+00:00', 'rows_pulled': 245, 'first_bar': '2026-10-01 00:00:00+00:00', 'last_bar': '2026-11-10 16:00:00+00:00'}
